# Weighted Ensemble

A stronger ADME/Tox-style ensemble that combines multiple tabular models and learns non-negative blending weights on the validation split.

In [3]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.optimize import minimize
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.metrics import mean_absolute_error

PROJECT_ROOT = Path('../..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.modeling import artifact_paths, load_tabular_arrays, plot_pred_vs_real, regression_metrics

In [4]:
RUN_ID = 'weighted_ensemble'
RANDOM_STATE = 42

#load data
X_train, X_valid, X_test, y_train, y_valid, y_test, feature_names = load_tabular_arrays('../../data/processed')
X_train = X_train.astype(np.float32)
X_valid = X_valid.astype(np.float32)
X_test = X_test.astype(np.float32)

#output paths where to find things!
paths = artifact_paths(Path.cwd(), RUN_ID, '.joblib', include_metadata=True)
paths['weights'] = Path.cwd() / 'outcome' / 'metadata' / f'{RUN_ID}_weights.csv'
paths['base_predictions'] = Path.cwd() / 'outcome' / 'predictions' / f'{RUN_ID}_base_predictions.csv'
for path in paths.values():
    path.parent.mkdir(parents=True, exist_ok=True)

#to visualize the data distribution
print(f'Train: {X_train.shape}, valid: {X_valid.shape}, test: {X_test.shape}')

Train: (5169, 1813), valid: (738, 1813), test: (1478, 1813)


In [5]:
#model preparation
def make_estimators():
    return {
        'ridge': RidgeCV(alphas=np.logspace(-3, 3, 13)),
        'elastic_net': ElasticNetCV(
            l1_ratio=[0.05, 0.1, 0.25, 0.5, 0.8],
            alphas=np.logspace(-4, 1, 12),
            cv=5,
            max_iter=20_000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        'random_forest': RandomForestRegressor(
            n_estimators=600,
            max_features='sqrt',
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        'extra_trees': ExtraTreesRegressor(
            n_estimators=800,
            max_features='sqrt',
            min_samples_leaf=1,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        'hist_gradient_boosting': HistGradientBoostingRegressor(
            learning_rate=0.04,
            max_iter=700,
            max_leaf_nodes=31,
            l2_regularization=0.05,
            early_stopping=True,
            random_state=RANDOM_STATE,
        ),
        'xgboost': xgb.XGBRegressor(
            n_estimators=800,
            learning_rate=0.035,
            max_depth=5,
            min_child_weight=2,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=1.5,
            objective='reg:squarederror',
            tree_method='hist',
            device='cuda',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }

#model management alligned
def fit_estimator(name, estimator, X_fit, y_fit, X_eval=None, y_eval=None):
    if name == 'xgboost' and X_eval is not None:
        try:
            estimator.fit(X_fit, y_fit, eval_set=[(X_eval, y_eval)], verbose=False)
        except Exception as exc:
            warnings.warn(f'XGBoost CUDA failed; retrying on CPU. Details: {exc}')
            estimator.set_params(device='cpu')
            estimator.fit(X_fit, y_fit, eval_set=[(X_eval, y_eval)], verbose=False)
    else:
        estimator.fit(X_fit, y_fit)
    return estimator

In [6]:
#train base models and collect predictions for the ensemble weights optimization
base_estimators = make_estimators()
fitted_for_weights = {}
valid_predictions = {}

for name, estimator in base_estimators.items():
    print(f'Training {name}...')
    fitted = fit_estimator(name, clone(estimator), X_train, y_train, X_valid, y_valid)
    preds = np.asarray(fitted.predict(X_valid)).ravel()
    fitted_for_weights[name] = fitted
    valid_predictions[name] = preds
    metrics = regression_metrics(y_valid, preds)
    print(f"  valid MAE={metrics['mae']:.4f} RMSE={metrics['rmse']:.4f} R2={metrics['r2']:.4f}")

valid_pred_matrix = np.column_stack([valid_predictions[name] for name in base_estimators])

Training ridge...
  valid MAE=0.5763 RMSE=0.7722 R2=0.2201
Training elastic_net...


c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.489e-01, tolerance: 3.532e-01
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.306e-01, tolerance: 3.532e-01
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the f

  valid MAE=0.5610 RMSE=0.7439 R2=0.2764
Training random_forest...
  valid MAE=0.5229 RMSE=0.6942 R2=0.3698
Training extra_trees...
  valid MAE=0.5117 RMSE=0.6831 R2=0.3898
Training hist_gradient_boosting...
  valid MAE=0.5018 RMSE=0.6816 R2=0.3924
Training xgboost...
  valid MAE=0.4993 RMSE=0.6801 R2=0.3952


c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\xgboost\core.py:751: UserWarning: [16:34:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [ ]:
#optimize ensemble weights using constrained optimization
def weighted_prediction(weights, pred_matrix):
    weights = np.asarray(weights, dtype=float)
    return pred_matrix @ weights

def objective(weights):
    return mean_absolute_error(y_valid, weighted_prediction(weights, valid_pred_matrix))


n_models = valid_pred_matrix.shape[1]
initial_weights = np.full(n_models, 1.0 / n_models)
constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
bounds = [(0.0, 1.0)] * n_models

result = minimize(
    objective,
    initial_weights,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-10},
)

if result.success:
    weights = result.x
else:
    warnings.warn(f'Weight optimization failed: {result.message}. Falling back to equal weights.')
    weights = initial_weights

weights = weights / weights.sum()
weight_table = pd.DataFrame({'model': list(base_estimators), 'weight': weights})
display(weight_table.sort_values('weight', ascending=False))

valid_ensemble = weighted_prediction(weights, valid_pred_matrix)
regression_metrics(y_valid, valid_ensemble)

,model,weight
5,xgboost,4.687174e-01
4,hist_gradient_boosting,4.079768e-01
3,extra_trees,8.286869e-02
0,ridge,4.043711e-02
1,elastic_net,3.314918e-17
2,random_forest,2.883040e-18


{'rmse': 0.6738036269927864,
 'mae': 0.49612528088469854,
 'r2': 0.40628078681717605}

In [8]:
#final fit on train + validation, then evaluate once on the held-out test set.
X_full = np.vstack([X_train, X_valid])
y_full = np.concatenate([y_train, y_valid])

final_estimators = {}
test_predictions = {}

for name, estimator in make_estimators().items():
    print(f'Final training {name}...')
    final_model = fit_estimator(name, clone(estimator), X_full, y_full, X_valid, y_valid)
    final_estimators[name] = final_model
    test_predictions[name] = np.asarray(final_model.predict(X_test)).ravel()

test_pred_matrix = np.column_stack([test_predictions[name] for name in base_estimators])
ensemble_predictions = weighted_prediction(weights, test_pred_matrix)
test_metrics = regression_metrics(y_test, ensemble_predictions)
print(f"[Weighted Ensemble] RMSE: {test_metrics['rmse']:.4f} | MAE: {test_metrics['mae']:.4f} | R2: {test_metrics['r2']:.4f}")
test_metrics

Final training ridge...
Final training elastic_net...


c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.213e-01, tolerance: 3.904e-01
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.623e+00, tolerance: 4.149e-01
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the f

Final training random_forest...
Final training extra_trees...
Final training hist_gradient_boosting...
Final training xgboost...
[Weighted Ensemble] RMSE: 0.8667 | MAE: 0.6011 | R2: 0.3296


{'rmse': 0.8667441006965644,
 'mae': 0.6010887210524748,
 'r2': 0.3296055440819238}

In [9]:
#save everything for analysis and reproducibility
base_prediction_df = pd.DataFrame({'real': y_test, 'ensemble': ensemble_predictions})
for name in base_estimators:
    base_prediction_df[name] = test_predictions[name]

base_prediction_df[['real', 'ensemble']].rename(columns={'ensemble': 'prediction'}).to_csv(paths['predictions'], index=False)
base_prediction_df.to_csv(paths['base_predictions'], index=False)
weight_table.to_csv(paths['weights'], index=False)
plot_pred_vs_real(paths['pred_vs_real'], y_test, ensemble_predictions, 'Weighted Ensemble: Predicted vs Real')

#joblib.dump({'models': final_estimators, 'weights': dict(zip(base_estimators.keys(), weights)), 'feature_names': feature_names}, paths['model'])

metadata = {
    'run_id': RUN_ID,
    'metrics': test_metrics,
    'validation_metrics': regression_metrics(y_valid, valid_ensemble),
    'weights': dict(zip(base_estimators.keys(), map(float, weights))),
}
with paths['metadata'].open('w', encoding='utf-8') as fp:
    json.dump(metadata, fp, indent=2)

paths

{'model': WindowsPath('c:/Users/Tommaso/Documents/MEGAR2D2/Documenti/SUPSI/BACHELOR/3_anno_bech/primaverile/M-D6310ZE-MLDD-25-26/toxicity-prediction/notebooks/ml-models/outcome/models/weighted_ensemble_model.joblib'),
 'predictions': WindowsPath('c:/Users/Tommaso/Documents/MEGAR2D2/Documenti/SUPSI/BACHELOR/3_anno_bech/primaverile/M-D6310ZE-MLDD-25-26/toxicity-prediction/notebooks/ml-models/outcome/predictions/weighted_ensemble_predictions.csv'),
 'pred_vs_real': WindowsPath('c:/Users/Tommaso/Documents/MEGAR2D2/Documenti/SUPSI/BACHELOR/3_anno_bech/primaverile/M-D6310ZE-MLDD-25-26/toxicity-prediction/notebooks/ml-models/outcome/weighted_ensemble_pred_vs_real.png'),
 'metadata': WindowsPath('c:/Users/Tommaso/Documents/MEGAR2D2/Documenti/SUPSI/BACHELOR/3_anno_bech/primaverile/M-D6310ZE-MLDD-25-26/toxicity-prediction/notebooks/ml-models/outcome/metadata/weighted_ensemble_metadata.json'),
 'weights': WindowsPath('c:/Users/Tommaso/Documents/MEGAR2D2/Documenti/SUPSI/BACHELOR/3_anno_bech/primav